## Harvester PoC

In [37]:
import torch
import pyarrow as pa
import pyarrow.parquet as pq
from transformers import AutoModelForCausalLM, AutoTokenizer

import tqdm as notebook_tqdm
from datetime import datetime
from pathlib import Path

OUTPUT_PATH = Path("/workspace/data/output")
TODAY_ID = datetime.today().strftime('%Y%m%d%M%S')
TODAY_DIR = OUTPUT_PATH / TODAY_ID
RUN_OUTPUT_PATH = TODAY_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "Qwen/Qwen2.5-7B-Instruct"
LAYER = 20

In [38]:
prompt = """
tell me what is so unique in Qwen2.5-7B-Instruct?
"""

In [39]:
tok = AutoTokenizer.from_pretrained(MODEL)
m = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    device_map="cuda"
)

# --- (a) generate a response ---

msgs = [
    {
        "role": "user",
        "content": prompt
    }
]

enc = tok.apply_chat_template(
    msgs,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

out = m.generate(
    **enc,                     # unpacks input_ids + attention_mask
    max_new_tokens=100,
    do_sample=True,
    temperature=1.0,
)

full = out[0]                                   # prompt + response token ids

prompt_len = enc["input_ids"].shape[1]
print(tok.decode(full[prompt_len:], skip_special_tokens=True))

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 136.86it/s]


Qwen-2.5-7B-Instruct is a large language model developed by Alibaba Cloud that has been fine-tuned specifically for instruction-following tasks. Here are some of the key features and unique aspects of this model:

1. **Fine-Tuning for Instruction-Following**: Unlike general-purpose models, Qwen-2.5-7B-Instruct has been specifically fine-tuned to follow instructions accurately. This means it's better at understanding and executing tasks as specified in the


In [40]:
# --- (b) one forward pass over the full sequence to expose hidden states ---
with torch.no_grad():
    hs = m(full.unsqueeze(0), output_hidden_states=True).hidden_states[LAYER][0]

# --- (c) inspect tokens and choose ---
for i, t in enumerate(full.tolist()):
    print(i, repr(tok.decode([t])))

CHOSEN = int(input("token index to verbalize: "))
vec = hs[CHOSEN]

## Write
activation_filename = f"{TODAY_ID}_activation.parquet"
prompt_filename = f"{TODAY_ID}_prompt.txt"

pq.write_table(
    pa.table({"activation_vector": [vec.float().cpu().tolist()]}),
    TODAY_DIR / activation_filename
)

with open(TODAY_DIR / prompt_filename, "w") as f:
    f.write(prompt)


print(f"saved token {CHOSEN} ({tok.decode([full[CHOSEN]])!r}) -> {TODAY_DIR / activation_filename}")

0 '<|im_start|>'
1 'system'
2 '\n'
3 'You'
4 ' are'
5 ' Q'
6 'wen'
7 ','
8 ' created'
9 ' by'
10 ' Alibaba'
11 ' Cloud'
12 '.'
13 ' You'
14 ' are'
15 ' a'
16 ' helpful'
17 ' assistant'
18 '.'
19 '<|im_end|>'
20 '\n'
21 '<|im_start|>'
22 'user'
23 '\n\n'
24 'tell'
25 ' me'
26 ' what'
27 ' is'
28 ' so'
29 ' unique'
30 ' in'
31 ' Q'
32 'wen'
33 '2'
34 '.'
35 '5'
36 '-'
37 '7'
38 'B'
39 '-In'
40 'struct'
41 '?\n'
42 '<|im_end|>'
43 '\n'
44 '<|im_start|>'
45 'assistant'
46 '\n'
47 'Q'
48 'wen'
49 '-'
50 '2'
51 '.'
52 '5'
53 '-'
54 '7'
55 'B'
56 '-In'
57 'struct'
58 ' is'
59 ' a'
60 ' large'
61 ' language'
62 ' model'
63 ' developed'
64 ' by'
65 ' Alibaba'
66 ' Cloud'
67 ' that'
68 ' has'
69 ' been'
70 ' fine'
71 '-t'
72 'uned'
73 ' specifically'
74 ' for'
75 ' instruction'
76 '-follow'
77 'ing'
78 ' tasks'
79 '.'
80 ' Here'
81 ' are'
82 ' some'
83 ' of'
84 ' the'
85 ' key'
86 ' features'
87 ' and'
88 ' unique'
89 ' aspects'
90 ' of'
91 ' this'
92 ' model'
93 ':\n\n'
94 '1'
95 '.'
96 ' **'
9

In [3]:
t = pq.read_table('/workspace/chosen.parquet')
print(t.schema)

activation_vector: list<element: double>
  child 0, element: double
